# SpendShield — Controlled ML Error Analysis

This research-only notebook audits the existing synthetic multiclass baseline.
It does not claim fraud detection performance and does not modify the dataset,
generator, backend, database, payment APIs, or production state.


## Scope and protocol

The analysis reproduces the stored Gaussian Naive Bayes baseline, uses the
existing timestamp-based train/validation/test partitions, reports validation
and final test errors, and reviews leakage, class imbalance, and generator
shortcuts. The test set is not used for fitting or model selection.


In [1]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "ml").is_dir() and (candidate / "data" / "synthetic").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.error_analysis import run_error_analysis

SOURCE_DIR = ROOT / "data" / "synthetic"
FEATURE_DIR = SOURCE_DIR / "features"
BASELINE_RESULT = SOURCE_DIR / "baseline" / "research_baseline_results.json"
OUTPUT_DIR = SOURCE_DIR / "error_analysis"

summary = run_error_analysis(SOURCE_DIR, FEATURE_DIR, BASELINE_RESULT, OUTPUT_DIR)
print(json.dumps({
    "valid": summary["valid"],
    "reproduction_valid": summary["reproduction_valid"],
    "leakage_review_valid": summary["leakage_review_valid"],
    "decision": summary["decision"],
}, indent=2))


{
  "valid": true,
  "reproduction_valid": true,
  "leakage_review_valid": true,
  "decision": "DATASET_OR_GENERATOR_REVIEW_REQUIRED_FIRST"
}


In [2]:
print("Validation accuracy:", summary["validation_metrics"]["accuracy"])
print("Validation macro F1:", summary["validation_metrics"]["macro_f1"])
print("Test accuracy:", summary["test_metrics"]["accuracy"])
print("Test macro F1:", summary["test_metrics"]["macro_f1"])
print("Test weighted F1:", summary["test_metrics"]["weighted_f1"])


Validation accuracy: 0.868332
Validation macro F1: 0.651481
Test accuracy: 0.862826
Test macro F1: 0.617953
Test weighted F1: 0.850388


In [3]:
def read_csv(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

test_metrics = summary["test_metrics"]
for class_name, metrics in test_metrics["per_class"].items():
    print(
        f"{class_name}: support={metrics['support']}, "
        f"precision={metrics['precision']}, recall={metrics['recall']}, f1={metrics['f1']}"
    )


normal: support=1132, precision=0.891736, recall=0.95318, f1=0.921435
synthetic_behavior_deviation: support=36, precision=0.210526, recall=0.222222, f1=0.216216
synthetic_combined_pattern: support=19, precision=0.4375, recall=0.368421, f1=0.4
synthetic_high_amount: support=114, precision=0.738739, recall=0.719298, f1=0.728889
synthetic_rapid_repeat: support=52, precision=1.0, recall=1.0, f1=1.0
synthetic_unusual_time: support=105, precision=0.967742, recall=0.285714, f1=0.441176


In [4]:
misclassified = read_csv(OUTPUT_DIR / "misclassified_test_records.csv")
error_counts = {}
for row in misclassified:
    error_counts[row["error_type"]] = error_counts.get(row["error_type"], 0) + 1
print("Bounded misclassified records exported:", len(misclassified))
print("Error groups:", error_counts)
print("First five safe inspection rows:")
for row in misclassified[:5]:
    print({key: row[key] for key in ("synthetic_transaction_id", "actual_target", "predicted_target", "error_type", "error_tags")})


Bounded misclassified records exported: 200
Error groups: {'confusion_between_synthetic_scenarios': 16, 'false_negative_for_synthetic_pattern': 131, 'false_positive_against_normal': 53}
First five safe inspection rows:
{'synthetic_transaction_id': 'syn-transaction-008543', 'actual_target': 'synthetic_behavior_deviation', 'predicted_target': 'synthetic_high_amount', 'error_type': 'confusion_between_synthetic_scenarios', 'error_tags': 'rare_class_miss'}
{'synthetic_transaction_id': 'syn-transaction-008545', 'actual_target': 'synthetic_behavior_deviation', 'predicted_target': 'normal', 'error_type': 'false_negative_for_synthetic_pattern', 'error_tags': 'rare_class_miss'}
{'synthetic_transaction_id': 'syn-transaction-008562', 'actual_target': 'synthetic_high_amount', 'predicted_target': 'normal', 'error_type': 'false_negative_for_synthetic_pattern', 'error_tags': ''}
{'synthetic_transaction_id': 'syn-transaction-008592', 'actual_target': 'normal', 'predicted_target': 'synthetic_high_amount

In [5]:
class_report = json.loads((OUTPUT_DIR / "class_distribution_report.json").read_text(encoding="utf-8"))
for split_name in ("full", "train", "validation", "test"):
    split = class_report[split_name]
    print(split_name, "rows=", split["row_count"], "majority=", split["majority_class"], "ratio=", split["majority_to_minority_ratio"])

feature_summary = read_csv(OUTPUT_DIR / "feature_distribution_summary.csv")
print("Feature distribution rows:", len(feature_summary), "split values:", sorted({row["split"] for row in feature_summary}))


full rows= 10000 majority= normal ratio= 76.0
train rows= 7061 majority= normal ratio= 98.962963
validation rows= 1481 majority= normal ratio= 41.62963
test rows= 1458 majority= normal ratio= 59.578947
Feature distribution rows: 48 split values: ['validation']


In [6]:
shortcut_review = json.loads((OUTPUT_DIR / "shortcut_review.json").read_text(encoding="utf-8"))
leakage_review = json.loads((OUTPUT_DIR / "leakage_review.json").read_text(encoding="utf-8"))
print("Shortcut status:", shortcut_review["status"])
print("Generator modified:", shortcut_review["generator_was_modified"])
print("Leakage review valid:", leakage_review["valid"])
print("Decision: DATASET_OR_GENERATOR_REVIEW_REQUIRED_FIRST")


Shortcut status: REVIEW_REQUIRED_BEFORE_COMPLEX_MODEL
Generator modified: False
Leakage review valid: True
Decision: DATASET_OR_GENERATOR_REVIEW_REQUIRED_FIRST


## Conclusion

The baseline is reproducible and the leakage/temporal checks pass. The
analysis does not justify a more complex model yet: rapid-repeat behavior is
perfectly recovered on the held-out test split, while unusual-time and
behavior-deviation cases remain difficult and the combined class is rare.
The next research step is dataset/generator review, not model escalation.
